# School List from Open Data

Builds a clean list of schools for a city (default: Kolkata) using **OpenStreetMap** via the Overpass API, then cleans and de-duplicates it with pandas. My original notebook tried to scrape HTML tables from arbitrary websites; this version uses an open dataset instead and only optionally reads tables from pages whose robots.txt allows it. Data © OpenStreetMap contributors (ODbL).

**How to run**
1. Open in Google Colab and run the cells in order.
2. Change `CITY_NAME` if you want another city.
3. Download `schools_clean.csv` from the Colab file panel.

**Responsible use**: only public institution-level fields (name, address, website, public phone) are kept, no personal emails or staff data. Respect each site's robots.txt and terms, and credit your sources.

_Rewritten from my original Colab notebook: `school list`._

## 1. Settings

In [ ]:
CITY_NAME = "Kolkata"

## 2. Helper: call the Overpass API with retries

In [ ]:
import time
import requests
import pandas as pd

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": "osm-practice-notebook/1.0 (educational use)"}

def run_overpass(query, retries=4, wait=30):
    """Send a query to Overpass. On errors / rate limits, wait longer each time and retry."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=200)
            if r.status_code == 200:
                return r.json()["elements"]
            print(f"Attempt {attempt}: HTTP {r.status_code}")
        except requests.RequestException as e:
            print(f"Attempt {attempt}: {e}")
        time.sleep(wait * attempt)  # simple backoff: 30s, 60s, 90s ...
    raise RuntimeError("Overpass request failed after all retries")

## 3. Download schools inside the city boundary

In [ ]:
query = f"""
[out:json][timeout:180];
area["name"="{CITY_NAME}"]["boundary"="administrative"]->.city;
nwr["amenity"="school"](area.city);
out center tags;
"""
elements = run_overpass(query)
print("Schools found:", len(elements))

In [ ]:
rows = []
for el in elements:
    t = el.get("tags", {})
    rows.append({
        "school_name": t.get("name", ""),
        "operator": t.get("operator", ""),
        "operator_type": t.get("operator:type", ""),
        "address": " ".join(x for x in [t.get("addr:housenumber", ""), t.get("addr:street", "")] if x),
        "postcode": t.get("addr:postcode", ""),
        "website": t.get("website") or t.get("contact:website", ""),
        "phone": t.get("phone") or t.get("contact:phone", ""),
        "source_name": "OpenStreetMap",
        "source_url": f"https://www.openstreetmap.org/{el['type']}/{el['id']}",
    })
raw = pd.DataFrame(rows)
raw.head()

## 4. (Optional) Add a table from a public web page

If you know a page (for example a government open-data page) with a table of schools, add its URL. The code checks `robots.txt` first and waits between requests.

In [ ]:
from io import StringIO
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

EXTRA_URLS = []   # e.g. ["https://example.gov/schools-table"]

def allowed_by_robots(url):
    parts = urlparse(url)
    rp = RobotFileParser(f"{parts.scheme}://{parts.netloc}/robots.txt")
    try:
        rp.read()
    except Exception:
        return False
    return rp.can_fetch(HEADERS["User-Agent"], url)

extra_tables = []
for url in EXTRA_URLS:
    if not allowed_by_robots(url):
        print("Skipping (robots.txt):", url)
        continue
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    for table in pd.read_html(StringIO(r.text)):
        table.columns = [str(c).strip().lower() for c in table.columns]
        name_col = next((c for c in table.columns if "name" in c or "school" in c), None)
        if name_col:
            extra_tables.append(pd.DataFrame({
                "school_name": table[name_col].astype(str),
                "source_name": "web table",
                "source_url": url,
            }))
    time.sleep(5)

if extra_tables:
    raw = pd.concat([raw] + extra_tables, ignore_index=True)
print("Rows before cleaning:", len(raw))

## 5. Clean and de-duplicate

In [ ]:
import re

raw = raw.fillna("")

def normalize_name(name):
    name = str(name).lower()
    name = re.sub(r"[^a-z0-9 ]", " ", name)
    return re.sub(r"\s+", " ", name).strip()

raw["name_key"] = raw["school_name"].apply(normalize_name)
clean = raw[raw["name_key"] != ""].drop_duplicates(subset="name_key").drop(columns="name_key")
print("Rows after cleaning:", len(clean))
clean.head()

## 6. Save

In [ ]:
clean.to_csv("schools_clean.csv", index=False)
print("Saved schools_clean.csv")